In [39]:
import json
import os

docs_path = "/Users/wnowogorski/PycharmProjects/CHAT_AGH/web_scraping/downloads/documents.json"

with open(docs_path) as json_file:
    data = json.load(json_file)

length_limit = 2000


In [40]:
data = [d for d in data if len(d["page_content"]) > length_limit]
len(data)

0

In [41]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages.utils import get_buffer_string
from langchain_core.runnables import Runnable

from chat_graph.agents.base_agent import BaseAgent

PROMPT = """
You are a helpful assistant tasked with generating question-answer pairs based on a given document. Your goal is to simulate realistic user questions that can be answered using the content of the document. The questions should vary in difficulty and type (e.g., factual, reasoning, summarization, etc.).

Instructions:

- Read the document carefully.
- Generate 3 to 5 question-answer pairs that are directly answerable from the document.
- Do not ask for the same thing twice, do not duplicate questions.

Questions should be:
- Varied in type: include factual, inferential, summarization, and "why/how" questions.
- Natural: as if asked by a curious person who just read the document or is looking for information it contains.
- Diverse in difficulty: include both simple and more complex ones.

Answers must be:
- Grounded strictly in the document.
- Comprehensive, contain all relevant information but grounded in the source.

Use the phrasing and terminology of the source when possible. Use polish language.
Formatting:

Output a JSON array of QA objects. Each object should include:

"question": the generated question
"answer": the answer
"difficulty": one of "easy", "medium", "hard"
"question_type": one of "factual", "reasoning", "summarization", "why/how"

Output Example:
{{
"questions": [
      {{
        "question": "What is the main purpose of the regulation introduced in 2021?",
        "answer": "To improve data transparency and enforce stricter privacy standards across the EU.",
        "difficulty": "medium",
        "question_type": "factual"
      }},
      {{
        "question": "Why did the council decide to delay the implementation of the policy?",
        "answer": "Because several member states expressed concern over the lack of infrastructure to support it.",
        "difficulty": "hard",
        "question_type": "why/how"
      }},
      {{
        "question": "Summarize the key responsibilities of the oversight committee.",
        "answer": "The oversight committee is responsible for monitoring compliance, reviewing reports, and advising on policy adjustments.",
        "difficulty": "medium",
        "question_type": "summarization"
      }}
    ]
}}


Input Document:
{DOCUMENT}

Your output:


"""
from chat_graph.utils import retry_on_exception
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class BenchmarkGenerationOutput(BaseModel):
    question: str = Field(..., description="Question")
    answer: str = Field(..., description="Answer")
    difficulty: str = Field(..., description="Difficulty")
    question_type: str = Field(..., description="Question Type")

from typing import List
class BenchmarkGenerationOutputList(BaseModel):
    questions: List[BenchmarkGenerationOutput] = Field(..., description="List of QA pairs")

class BenchmarkGenerationAgent(BaseAgent):
    def __init__(self, **kwargs):
        super().__init__(prompt_template=PROMPT, **kwargs)
        self.output_parser = PydanticOutputParser(pydantic_object=BenchmarkGenerationOutputList)

        self.prompt = PromptTemplate(
            input_variables=["DOCUMENT"],
            template=self.prompt_template
        )

        self.chain: Runnable = self.prompt | self.llm | self.output_parser

    @retry_on_exception(attempts=4, delay=5, backoff=3)
    def inference(self, document):
        response = self.chain.invoke({
            "DOCUMENT": document,
        })
        return response.questions



In [36]:
agent = BenchmarkGenerationAgent()

In [38]:
benchmark_dict = []

for doc in data[::-1]:
    source_url = doc["metadata"]["url"]
    print(source_url)
    content = doc["page_content"]
    response = agent.inference(document=content)
    questions = response
    for q in response:
        question = {
            "question": q.question,
            "answer": q.answer,
            "difficulty": q.difficulty,
            "question_type": q.question_type,
            "source_url": source_url,
        }
        benchmark_dict.append(question)

    with open("benchmarks/benchmark_agh_pl.json", "w") as json_file:
        json.dump(benchmark_dict, json_file)


https://www.miasteczko.agh.edu.pl/en/resident-zone/pricing.html#Basic
https://www.miasteczko.agh.edu.pl/en/resident-zone/pricing.html#Comfort
https://www.miasteczko.agh.edu.pl/en/resident-zone/pricing.html#Comfort+
https://www.miasteczko.agh.edu.pl/modules/DownloadManager/download.php?alias=zasady-uzytkowania-kluczy-systemowych-ms-agh-08-07-2024-3-1
https://www.miasteczko.agh.edu.pl/modules/DownloadManager/download.php?alias=regulamin-miasteczka
https://www.miasteczko.agh.edu.pl/strefa-mieszkanca/cennik.html#Podstawowy
https://www.miasteczko.agh.edu.pl/strefa-mieszkanca/cennik.html#Komfort
https://www.miasteczko.agh.edu.pl/strefa-mieszkanca/cennik.html#Komfort+
https://www.miasteczko.agh.edu.pl/en/pricing.html
https://www.miasteczko.agh.edu.pl/en/questions-and-answers.html
https://www.miasteczko.agh.edu.pl/en/about-us.html
https://www.miasteczko.agh.edu.pl/strefa-mieszkanca/cennik.html
https://www.miasteczko.agh.edu.pl/pl/horizontal/deklaracja-dostepnosci.html
https://www.miasteczko.ag

In [34]:
len(benchmark_dict)

74